In [ ]:
#org/A,B,C→crop,mask_org/A,B,Cに保存

import torch
from ultralytics import YOLO
from PIL import Image
import sys
import os
import cv2
import numpy as np
import glob

# -------------------------------------------------------------
# 1. 各種パスを指定 (★ここを変更してください)
# -------------------------------------------------------------
# 処理したい画像ファイルが入っているフォルダのパス
INPUT_IMAGE_DIR = '/home/data/1021_fasttest/org/A' 

# 学習済みモデルのパス
MODEL_PATH = '/home/YOLO/0708_maesyori/datasets/train/weights/best.pt'

# クロップ画像とマスクの保存先フォルダ
CROP_DIR = '/home/data/1021_fasttest/crop/A'
MASK_DIR = '/home/data/1021_fasttest/mask_org/A'

# -------------------------------------------------------------
# 2. モデルと保存先フォルダの準備
# -------------------------------------------------------------
# --- ここから下のコードは変更不要です ---
print("セグメンテーションモデルを読み込んでいます...")
try:
    model = YOLO(MODEL_PATH)
    print("モデルの読み込み完了。")
except Exception as e:
    print(f"\n[エラー] モデルの読み込みに失敗しました。")
    print(f"パスが正しいか確認してください: {MODEL_PATH}")
    print(f"詳細: {e}")
    sys.exit()

# 保存先ディレクトリを作成 (フォルダがなければ自動で作成)
os.makedirs(CROP_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)
print(f"クロップ画像の保存先: {CROP_DIR}")
print(f"マスク画像の保存先:   {MASK_DIR}")

# -------------------------------------------------------------
# 3. 指定フォルダ内の画像を順番に処理
# -------------------------------------------------------------
print("\n画像処理を開始します...")

# 対象とする画像の拡張子
allowed_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp']

# フォルダ内の画像ファイルリストを取得
image_paths = []
for ext in allowed_extensions:
    image_paths.extend(glob.glob(os.path.join(INPUT_IMAGE_DIR, ext)))

if not image_paths:
    print(f"\n[エラー] 指定されたフォルダに処理対象の画像が見つかりません。")
    print(f"パスを確認してください: {INPUT_IMAGE_DIR}")
    sys.exit()

print(f"{len(image_paths)} 件の画像ファイルを処理します。")

# 取得した画像パスでループ処理を開始
for i, image_path in enumerate(image_paths):
    print(f"\n--- [{i+1}/{len(image_paths)}] 処理中: {os.path.basename(image_path)} ---")
    
    # (A) 画像の読み込み
    try:
        img = Image.open(image_path).convert("RGB")
    except Exception as e:
        print(f"[警告] 画像の読み込みに失敗しました。このファイルをスキップします。詳細: {e}")
        continue # エラーが発生した場合は次の画像へ

    # (B) セグメンテーション実行
    results = model(img, retina_masks=True)
    result = results[0]

    # (C) クロップ画像とマスクを個別に保存
    base_filename = os.path.splitext(os.path.basename(image_path))[0]

    if result.masks is not None and result.boxes is not None:
        num_objects = len(result.masks.data)
        print(f"{num_objects}個の物体を検出しました。")
        original_img_np = np.array(img)

        # 検出された物体を一つずつループ
        for i in range(num_objects):
            # バウンディングボックス座標を取得
            bbox_tensor = result.boxes.xyxy[i]
            x1, y1, x2, y2 = map(int, bbox_tensor)
            
            # 1. 物体をクロップして保存
            cropped_object_rgb = original_img_np[y1:y2, x1:x2]
            cropped_object_bgr = cv2.cvtColor(cropped_object_rgb, cv2.COLOR_RGB2BGR)
            crop_filename = f"{base_filename}_object_{i}.png"
            crop_path = os.path.join(CROP_DIR, crop_filename)
            cv2.imwrite(crop_path, cropped_object_bgr)

            # 2. クロップ画像に合わせたマスクを生成して保存
            full_mask_canvas = np.zeros(original_img_np.shape[:2], dtype=np.uint8)
            mask_tensor = result.masks.data[i]
            mask_np = mask_tensor.cpu().numpy().astype(np.uint8)
            mask_resized = cv2.resize(mask_np, (original_img_np.shape[1], original_img_np.shape[0]))
            full_mask_canvas[mask_resized > 0] = 255
            
            cropped_mask = full_mask_canvas[y1:y2, x1:x2]
            
            mask_filename = f"{base_filename}_mask_{i}.png"
            mask_path = os.path.join(MASK_DIR, mask_filename)
            cv2.imwrite(mask_path, cropped_mask)

        print(f"-> 保存完了: {os.path.basename(image_path)}")
    else:
        print("-> 物体は検出されませんでした。")

print("\n===================================")
print("すべての画像の処理が完了しました。")
print("===================================")

In [ ]:
#mask_org/A,B,C→mask_to1/A,B,Cに保存

import os
import cv2
import numpy as np
from tqdm import tqdm

# --- 設定 ---
# 1. 処理したいマスク画像が入っているフォルダ
INPUT_DIR = '/home/data/1021_fasttest/mask_org/A'

# 2. 処理後のマスクを保存するフォルダ
OUTPUT_DIR = '/home/data/1021_fasttest/mask_to1/A'
# ---

def keep_largest_component(image: np.ndarray) -> np.ndarray:
    """
    二値画像を受け取り、最大の連結成分のみを残した画像を返す。
    """
    # 連結成分を解析
    # num_labels: 背景を含む連結成分の総数
    # labels: 各ピクセルがどの成分に属するかを示すラベル行列
    # stats: 各成分の統計情報（[x, y, width, height, area]）
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(image, connectivity=8)

    # 成分が1つ以下（背景のみ、または1つの塊のみ）の場合はそのまま返す
    if num_labels <= 2:
        return image

    # 背景（ラベル0）を除いた最大の成分を見つける
    # stats[1:, cv2.CC_STAT_AREA] で背景以外の全成分の面積を取得
    largest_component_label = np.argmax(stats[1:, cv2.CC_STAT_AREA]) + 1

    # 新しいマスクを生成
    # 最大の成分に属するラベルを持つピクセルのみを白(255)にする
    cleaned_mask = np.zeros_like(image)
    cleaned_mask[labels == largest_component_label] = 255
    
    return cleaned_mask

def process_masks_in_directory():
    """
    指定されたディレクトリ内のすべてのマスク画像を処理する。
    """
    # 保存先フォルダが存在しない場合は作成
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"入力フォルダ: '{INPUT_DIR}'")
    print(f"出力フォルダ: '{OUTPUT_DIR}'")

    # 処理対象の画像ファイルを取得
    try:
        image_files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if not image_files:
            print(f"エラー: 入力フォルダ '{INPUT_DIR}' に画像ファイルが見つかりません。")
            return
    except FileNotFoundError:
        print(f"エラー: 入力フォルダ '{INPUT_DIR}' が見つかりません。パスを確認してください。")
        return
        
    print(f"{len(image_files)} 件のマスク画像を処理します...")

    for filename in tqdm(image_files, desc="マスクをクリーンアップ中"):
        # 画像をグレースケールで読み込み
        image_path = os.path.join(INPUT_DIR, filename)
        mask_image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

        if mask_image is None:
            continue

        # 最大の連結成分のみを残す処理
        cleaned_image = keep_largest_component(mask_image)

        # 結果を保存
        output_path = os.path.join(OUTPUT_DIR, filename)
        cv2.imwrite(output_path, cleaned_image)

    print("すべての処理が完了しました。")


if __name__ == '__main__':
    process_masks_in_directory()

In [ ]:
#mask_to1/A,B,C→mask_large,mask_small/A,B,Cに保存
#50000ピクセル以下をmask_smallに，50000以下のcrop削除
#crop/A,B,Cの対応するcropを削除

import os
import cv2
import shutil
import numpy as np
from tqdm import tqdm

# --- 設定 ---
# 振り分けたいマスク画像が入っている元のフォルダ
SOURCE_DIR = '/home/data/1021_fasttest/mask_to1/A'

# 条件を満たした画像（大きい画像）の移動先フォルダ
DEST_DIR = '/home/data/1021_fasttest/mask_large/A'

# 条件から外れた画像（小さい画像）の移動先フォルダ
SMALL_DIR = '/home/data/1021_fasttest/mask_small/A'

# ★★★ 追加 ★★★
# 削除対象のクロップ画像が入っているフォルダ
CROP_DIR = '/home/data/1021_fasttest/crop/A'
# ★★★★★★★★

# 振り分けの基準となる面積のしきい値 (ピクセル単位)
AREA_THRESHOLD = 50000
# ---

def filter_and_cleanup_images():
    """
    固定のしきい値で画像を振り分け、小さい画像に対応するクロップ画像を削除する。
    """
    os.makedirs(DEST_DIR, exist_ok=True)
    os.makedirs(SMALL_DIR, exist_ok=True)

    try:
        filenames = [
            f for f in os.listdir(SOURCE_DIR) 
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp'))
        ]
    except FileNotFoundError:
        print(f"[エラー] 元フォルダが見つかりません: {SOURCE_DIR}")
        return

    if not filenames:
        print(f"'{SOURCE_DIR}' 内に処理対象の画像が見つかりませんでした。")
        return

    print(f"しきい値 {AREA_THRESHOLD} ピクセルに基づき、画像を振り分け、不要なクロップ画像を削除します...")
    
    deleted_count = 0
    for filename in tqdm(filenames, desc="画像を振り分け中"):
        source_path = os.path.join(SOURCE_DIR, filename)
        
        try:
            img = cv2.imread(source_path)
            if img is None:
                print(f"\n[警告] {filename} が読み込めませんでした。スキップします。")
                continue
                
            height, width, _ = img.shape
            area = height * width
            
            if area < AREA_THRESHOLD:
                # 小さい画像をSMALL_DIRへ移動
                shutil.copy(source_path, os.path.join(SMALL_DIR, filename))

                # ★★★ 追加: 対応するクロップ画像を削除する処理 ★★★
                # 1. マスクファイル名からクロップファイル名を生成
                #    (例: '..._mask_0.png' -> '..._object_0.png')
                crop_filename = filename.replace('_mask_', '_object_')
                
                # 2. 削除するクロップ画像のフルパスを作成
                crop_path_to_delete = os.path.join(CROP_DIR, crop_filename)
                
                # 3. ファイルが存在すれば削除
                if os.path.exists(crop_path_to_delete):
                    os.remove(crop_path_to_delete)
                    deleted_count += 1
                # ★★★★★★★★★★★★★★★★★★★★★★★★★★

            else:
                # 大きい画像をDEST_DIRへ移動
                shutil.copy(source_path, os.path.join(DEST_DIR, filename))

        except Exception as e:
            print(f"\n[エラー] {filename} の処理中に問題が発生しました: {e}")
            
    print("\n振り分けが完了しました。")
    print(f"  - 大きい画像の保存先: '{DEST_DIR}/'")
    print(f"  - 小さい画像の保存先: '{SMALL_DIR}/'")
    if deleted_count > 0:
        print(f"  - 削除されたクロップ画像: {deleted_count}枚")


if __name__ == '__main__':
    filter_and_cleanup_images()

In [ ]:
#mask_large/A,B,C→mask/A,B,Cに保存

import os
import cv2
import numpy as np
from tqdm import tqdm

# --- 設定 ---
# 1. 穴埋め処理をしたいマスク画像が入っているフォルダ
INPUT_DIR = '/home/data/1021_fasttest/mask_large/A'  # 対象フォルダ名に合わせて変更してください

# 2. 処理後のマスクを保存するフォルダ
OUTPUT_DIR = '/home/data/1021_fasttest/mask/A'
# ---

def fill_true_holes(image: np.ndarray) -> np.ndarray:
    """
    画像内の「完全に白で囲まれた黒い領域（真の穴）」のみを塗りつぶす。
    """
    # 念のため、画像を完全な二値（0と255）に変換
    _, binary_image = cv2.threshold(image, 127, 255, cv2.THRESH_BINARY)
    
    # 変更を加えるための画像のコピーを作成
    output_image = binary_image.copy()
    
    # 画像を反転させ、黒い領域を白い塊として扱う
    inverted_image = cv2.bitwise_not(binary_image)
    
    # 反転させた画像から、各黒領域の輪郭を見つける
    # cv2.RETR_EXTERNAL を使うことで、個々の塊（背景、穴）を別々に取得できる
    contours, _ = cv2.findContours(inverted_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # 画像の高さと幅を取得
    h, w = binary_image.shape[:2]
    
    # 各輪郭（黒い塊）をチェック
    for cnt in contours:
        # 輪郭のバウンディングボックス（輪郭を囲む最小の長方形）を取得
        x, y, rect_w, rect_h = cv2.boundingRect(cnt)
        
        # バウンディングボックスが画像の端に接しているか判定
        is_on_border = (x == 0) or (y == 0) or (x + rect_w == w) or (y + rect_h == h)
        
        # もし画像の端に接していなければ、それは「真の穴」である
        if not is_on_border:
            # その穴を白(255)で塗りつぶす
            cv2.drawContours(output_image, [cnt], -1, color=255, thickness=cv2.FILLED)
            
    return output_image

def process_masks_in_directory():
    """
    指定されたディレクトリ内のすべてのマスク画像の穴を埋める。
    """
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"入力フォルダ: '{INPUT_DIR}'")
    print(f"出力フォルダ: '{OUTPUT_DIR}'")

    try:
        image_files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if not image_files:
            print(f"エラー: 入力フォルダ '{INPUT_DIR}' に画像ファイルが見つかりません。")
            return
    except FileNotFoundError:
        print(f"エラー: 入力フォルダ '{INPUT_DIR}' が見つかりません。パスを確認してください。")
        return
        
    print(f"{len(image_files)} 件のマスク画像を処理します...")

    for filename in tqdm(image_files, desc="マスクの真の穴を充填中"):
        image_path = os.path.join(INPUT_DIR, filename)
        mask_image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

        if mask_image is None:
            continue

        # 真の穴を埋める処理
        filled_image = fill_true_holes(mask_image)

        # 結果を保存
        output_path = os.path.join(OUTPUT_DIR, filename)
        cv2.imwrite(output_path, filled_image)

    print("すべての処理が完了しました。")

if __name__ == '__main__':
    process_masks_in_directory()

In [ ]:
#crop/A,B,Cとmask/A,B,C→combined/A,B,Cに保存

import os
import cv2
import numpy as np
from tqdm.notebook import tqdm

# --- 設定 ---
# 1. 元となるクロップ画像が入っているフォルダ
CROP_IMAGE_DIR = '/home/data/1021_fasttest/crop/A'

# 2. マスク画像が入っているフォルダ
MASK_IMAGE_DIR = '/home/data/1021_fasttest/mask/A'

# 3. マスク適用後の画像の保存先フォルダ
OUTPUT_DIR = '/home/data/1021_fasttest/combined/A'
# ---

def apply_masks_to_images():
    """
    クロップ画像フォルダとマスク画像フォルダから対応するファイルを読み込み、
    マスクを適用した画像を生成して保存する。
    """
    # 保存先フォルダがなければ作成
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"入力(クロップ画像): '{CROP_IMAGE_DIR}'")
    print(f"入力(マスク画像)  : '{MASK_IMAGE_DIR}'")
    print(f"出力先            : '{OUTPUT_DIR}'")

    try:
        # クロップ画像フォルダ内の画像ファイルリストを取得
        crop_files = [
            f for f in os.listdir(CROP_IMAGE_DIR) 
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]
    except FileNotFoundError:
        print(f"❌ [エラー] クロップ画像フォルダが見つかりません: {CROP_IMAGE_DIR}")
        return

    if not crop_files:
        print(f"📂 '{CROP_IMAGE_DIR}' 内に処理対象の画像が見つかりませんでした。")
        return

    print(f"\n🖼️  {len(crop_files)}枚の画像にマスクを適用します...")
    
    success_count = 0
    not_found_count = 0
    for crop_filename in tqdm(crop_files, desc="マスク適用中"):
        # 対応するマスクファイル名を生成
        # (例: '..._object_0.png' -> '..._mask_0.png')
        mask_filename = crop_filename.replace('_object_', '_mask_')
        
        # 各ファイルのフルパスを作成
        crop_path = os.path.join(CROP_IMAGE_DIR, crop_filename)
        mask_path = os.path.join(MASK_IMAGE_DIR, mask_filename)
        
        # 対応するマスクファイルが存在するかチェック
        if not os.path.exists(mask_path):
            # print(f"\n[警告] {crop_filename} に対応するマスクが見つかりません。スキップします。")
            not_found_count += 1
            continue
            
        try:
            # 画像とマスクを読み込む
            crop_image = cv2.imread(crop_path)
            # マスクはグレースケールで読み込む
            mask_image = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

            if crop_image is None or mask_image is None:
                continue

            # マスクを二値化（白か黒かにする）
            _, binary_mask = cv2.threshold(mask_image, 127, 255, cv2.THRESH_BINARY)
            
            # マスクを3チャンネルのカラー画像に変換して、元の画像とサイズを合わせる
            mask_3ch = cv2.cvtColor(binary_mask, cv2.COLOR_GRAY2BGR)

            # マスクが白い部分だけ元の画像を残し、黒い部分は黒にする
            # (bitwise_andは両方の画像でピクセル値が0でない部分だけを残す)
            masked_image = cv2.bitwise_and(crop_image, mask_3ch)

            # 結果を保存
            output_path = os.path.join(OUTPUT_DIR, crop_filename)
            cv2.imwrite(output_path, masked_image)
            success_count += 1
            
        except Exception as e:
            print(f"\n[エラー] {crop_filename} の処理中に問題が発生しました: {e}")

    print("\n--- 処理完了 ---")
    print(f"✅ 正常に処理された画像: {success_count}枚")
    if not_found_count > 0:
        print(f"⚠️  対応するマスクが見つからなかった画像: {not_found_count}枚")

# --- 関数の実行 ---
apply_masks_to_images()

In [ ]:
#IMG_2648_mask_0.png→IMG_6480_mask.png(crop，combinedも同様)

import os
from tqdm.notebook import tqdm

# --- 設定 ---
# 1. crop画像フォルダ
CROP_DIR = '/home/data/1021_fasttest/crop/A'

# 2. mask画像フォルダ
MASK_DIR = '/home/data/1021_fasttest/mask/A'

# 3. マスク適用済み画像（combined）フォルダ
COMBINED_DIR = '/home/data/1021_fasttest/combined/A'
# ---

def rename_files_in_folder(target_dir: str, suffix: str):
    """
    指定されたフォルダ内の画像ファイル名を新しいルールに従って一括変更する。
    """
    print(f"--- フォルダ '{os.path.basename(target_dir)}' の処理を開始 ---")
    
    try:
        # フォルダ内のファイルリストを取得
        filenames = [
            f for f in os.listdir(target_dir) 
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]
    except FileNotFoundError:
        print(f"❌ [エラー] フォルダが見つかりません: {target_dir}")
        return

    if not filenames:
        print("📂 処理対象の画像が見つかりませんでした。")
        return
        
    renamed_count = 0
    # プログレスバーを表示しながら処理
    for filename in tqdm(filenames, desc=f"{suffix} 画像をリネーム中"):
        try:
            # 拡張子とそれ以外（基底名）に分離 -> ('IMG_2642_object_0', '.png')
            base, ext = os.path.splitext(filename)
            
            # '_'で分割 -> ['IMG', '2642', 'object', '0']
            parts = base.split('_')
            
            # 元画像の番号（'2642'）と末尾の番号（'0'）を取得
            original_num_str = parts[1]
            index_str = parts[-1] # 末尾の要素を取得
            
            # 新しい名前の要素を組み立てる
            xxx = original_num_str[-3:] # 下3桁を取得 -> '642'
            y = index_str               # -> '0'
            
            # 新しいファイル名を生成
            new_filename = f"IMG_{xxx}{y}_{suffix}{ext}" # -> 'IMG_6420_crop.png'
            
            # 古いパスと新しいパスを定義
            old_path = os.path.join(target_dir, filename)
            new_path = os.path.join(target_dir, new_filename)
            
            # ファイル名を変更
            if old_path != new_path:
                os.rename(old_path, new_path)
                renamed_count += 1

        except IndexError:
            # ファイル名が 'IMG_xxxx_suffix_y' の形式でない場合はスキップ
            print(f"\n[警告] '{filename}' は期待される形式でないため、スキップしました。")
        except Exception as e:
            print(f"\n[エラー] '{filename}' の処理中に問題が発生しました: {e}")

    print(f"✅ {renamed_count}件のファイル名を変更しました。\n")

# --- メインの処理 ---
if __name__ == '__main__':
    # 各フォルダに対してリネーム処理を実行
    rename_files_in_folder(CROP_DIR, 'crop')
    rename_files_in_folder(MASK_DIR, 'mask')
    rename_files_in_folder(COMBINED_DIR, 'combined')
    
    print("すべてのリネーム処理が完了しました。")

ライブラリ

In [ ]:
import torch
from ultralytics import YOLO
from PIL import Image
import sys
import os
import cv2
import numpy as np
import glob
import shutil

# -----------------------------------------------------------------
# [ステップ1] セグメンテーションとクロップ
# -----------------------------------------------------------------
def segment_and_crop(input_image_dir: str, model_path: str, crop_dir: str, mask_dir: str):
    """
    YOLOモデルを使用して画像をセグメンテーションし、
    クロップ画像とマスク画像を保存する。
    """
    try:
        model = YOLO(model_path)
    except Exception as e:
        raise RuntimeError(f"モデルの読み込みに失敗しました: {model_path}. 詳細: {e}")

    os.makedirs(crop_dir, exist_ok=True)
    os.makedirs(mask_dir, exist_ok=True)

    allowed_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp']
    image_paths = []
    for ext in allowed_extensions:
        image_paths.extend(glob.glob(os.path.join(input_image_dir, ext)))

    if not image_paths:
        return # 処理対象画像がなくてもエラーにしない

    for image_path in image_paths:
        try:
            img = Image.open(image_path).convert("RGB")
        except Exception as e:
            continue # 画像読み込み失敗時はスキップ

        results = model(img, retina_masks=True)
        result = results[0]
        base_filename = os.path.splitext(os.path.basename(image_path))[0]

        if result.masks is not None and result.boxes is not None:
            num_objects = len(result.masks.data)
            original_img_np = np.array(img)

            for i in range(num_objects):
                try:
                    bbox_tensor = result.boxes.xyxy[i]
                    x1, y1, x2, y2 = map(int, bbox_tensor)
                    
                    # 1. クロップ
                    cropped_object_rgb = original_img_np[y1:y2, x1:x2]
                    cropped_object_bgr = cv2.cvtColor(cropped_object_rgb, cv2.COLOR_RGB2BGR)
                    crop_filename = f"{base_filename}_object_{i}.png"
                    crop_path = os.path.join(crop_dir, crop_filename)
                    cv2.imwrite(crop_path, cropped_object_bgr)

                    # 2. マスク
                    full_mask_canvas = np.zeros(original_img_np.shape[:2], dtype=np.uint8)
                    mask_tensor = result.masks.data[i]
                    mask_np = mask_tensor.cpu().numpy().astype(np.uint8)
                    mask_resized = cv2.resize(mask_np, (original_img_np.shape[1], original_img_np.shape[0]))
                    full_mask_canvas[mask_resized > 0] = 255
                    
                    cropped_mask = full_mask_canvas[y1:y2, x1:x2]
                    
                    mask_filename = f"{base_filename}_mask_{i}.png"
                    mask_path = os.path.join(mask_dir, mask_filename)
                    cv2.imwrite(mask_path, cropped_mask)
                except Exception as e:
                    continue # オブジェクトごとの処理でエラーが起きても続行


# -----------------------------------------------------------------
# [ステップ2] マスクのクリーンアップ（最大連結成分）
# -----------------------------------------------------------------
def _keep_largest_component(image: np.ndarray) -> np.ndarray:
    """
    二値画像を受け取り、最大の連結成分のみを残した画像を返す。(内部ヘルパー関数)
    """
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(image, connectivity=8)
    if num_labels <= 2:
        return image
    largest_component_label = np.argmax(stats[1:, cv2.CC_STAT_AREA]) + 1
    cleaned_mask = np.zeros_like(image)
    cleaned_mask[labels == largest_component_label] = 255
    return cleaned_mask

def clean_masks_keep_largest(input_dir: str, output_dir: str):
    """
    マスク画像の連結成分を解析し、最大の成分のみを残す。
    """
    os.makedirs(output_dir, exist_ok=True)
    
    try:
        image_files = [f for f in os.listdir(input_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    except FileNotFoundError:
        raise FileNotFoundError(f"入力フォルダが見つかりません: {input_dir}")
    
    if not image_files:
        return

    for filename in image_files:
        image_path = os.path.join(input_dir, filename)
        mask_image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

        if mask_image is None:
            continue

        cleaned_image = _keep_largest_component(mask_image)
        output_path = os.path.join(output_dir, filename)
        cv2.imwrite(output_path, cleaned_image)


# -----------------------------------------------------------------
# [ステップ3] 面積によるフィルタリングとクロップ削除
# -----------------------------------------------------------------
def filter_masks_by_area(
    source_mask_dir: str, 
    large_mask_dir: str, 
    small_mask_dir: str, 
    crop_dir: str, 
    area_threshold: int
):
    """
    マスク画像の面積に基づき画像を振り分け、
    小さいマスクに対応するクロップ画像を削除する。
    """
    os.makedirs(large_mask_dir, exist_ok=True)
    os.makedirs(small_mask_dir, exist_ok=True)

    try:
        filenames = [
            f for f in os.listdir(source_mask_dir) 
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp'))
        ]
    except FileNotFoundError:
        raise FileNotFoundError(f"元フォルダが見つかりません: {source_mask_dir}")

    if not filenames:
        return

    for filename in filenames:
        source_path = os.path.join(source_mask_dir, filename)
        
        try:
            img = cv2.imread(source_path)
            if img is None:
                continue
                
            height, width, _ = img.shape
            area = height * width
            
            if area < area_threshold:
                # 小さい画像をコピー
                shutil.copy(source_path, os.path.join(small_mask_dir, filename))
                
                # 対応するクロップ画像を削除
                crop_filename = filename.replace('_mask_', '_object_')
                crop_path_to_delete = os.path.join(crop_dir, crop_filename)
                
                if os.path.exists(crop_path_to_delete):
                    os.remove(crop_path_to_delete)
            else:
                # 大きい画像をコピー
                shutil.copy(source_path, os.path.join(large_mask_dir, filename))

        except Exception as e:
            continue # 処理中にエラーが起きても続行


# -----------------------------------------------------------------
# [ステップ4] マスクの穴埋め
# -----------------------------------------------------------------
def _fill_true_holes(image: np.ndarray) -> np.ndarray:
    """
    画像内の「完全に白で囲まれた黒い領域（真の穴）」のみを塗りつぶす。(内部ヘルパー関数)
    """
    _, binary_image = cv2.threshold(image, 127, 255, cv2.THRESH_BINARY)
    output_image = binary_image.copy()
    inverted_image = cv2.bitwise_not(binary_image)
    contours, _ = cv2.findContours(inverted_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    h, w = binary_image.shape[:2]
    
    for cnt in contours:
        x, y, rect_w, rect_h = cv2.boundingRect(cnt)
        is_on_border = (x == 0) or (y == 0) or (x + rect_w == w) or (y + rect_h == h)
        
        if not is_on_border:
            cv2.drawContours(output_image, [cnt], -1, color=255, thickness=cv2.FILLED)
    return output_image

def fill_mask_holes(input_dir: str, output_dir: str):
    """
    マスク画像内の「真の穴」（画像端に接していない穴）を塗りつぶす。
    """
    os.makedirs(output_dir, exist_ok=True)

    try:
        image_files = [f for f in os.listdir(input_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    except FileNotFoundError:
        raise FileNotFoundError(f"入力フォルダが見つかりません: {input_dir}")
    
    if not image_files:
        return

    for filename in image_files:
        image_path = os.path.join(input_dir, filename)
        mask_image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

        if mask_image is None:
            continue

        filled_image = _fill_true_holes(mask_image)
        output_path = os.path.join(output_dir, filename)
        cv2.imwrite(output_path, filled_image)


# -----------------------------------------------------------------
# [ステップ5] マスクの適用
# -----------------------------------------------------------------
def apply_mask_to_crop(crop_image_dir: str, mask_image_dir: str, output_dir: str):
    """
    クロップ画像にマスク画像を適用し、背景を黒にする。
    """
    os.makedirs(output_dir, exist_ok=True)

    try:
        crop_files = [
            f for f in os.listdir(crop_image_dir) 
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]
    except FileNotFoundError:
        raise FileNotFoundError(f"クロップ画像フォルダが見つかりません: {crop_image_dir}")

    if not crop_files:
        return
    
    for crop_filename in crop_files:
        mask_filename = crop_filename.replace('_object_', '_mask_')
        
        crop_path = os.path.join(crop_image_dir, crop_filename)
        mask_path = os.path.join(mask_image_dir, mask_filename)
        
        if not os.path.exists(mask_path):
            continue
            
        try:
            crop_image = cv2.imread(crop_path)
            mask_image = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

            if crop_image is None or mask_image is None:
                continue

            _, binary_mask = cv2.threshold(mask_image, 127, 255, cv2.THRESH_BINARY)
            mask_3ch = cv2.cvtColor(binary_mask, cv2.COLOR_GRAY2BGR)
            masked_image = cv2.bitwise_and(crop_image, mask_3ch)

            output_path = os.path.join(output_dir, crop_filename)
            cv2.imwrite(output_path, masked_image)
            
        except Exception as e:
            continue


# -----------------------------------------------------------------
# [ステップ6] ファイル名の変更
# -----------------------------------------------------------------
def _rename_files_in_folder(target_dir: str, suffix: str):
    """
    指定フォルダ内のファイル名をルールに従い変更する。(内部ヘルパー関数)
    """
    try:
        filenames = [
            f for f in os.listdir(target_dir) 
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]
    except FileNotFoundError:
        return # フォルダがなくてもエラーにしない

    if not filenames:
        return
        
    for filename in filenames:
        try:
            base, ext = os.path.splitext(filename)
            parts = base.split('_')
            
            # 'IMG_2642_object_0' のような形式を想定
            original_num_str = parts[1] # '2642'
            index_str = parts[-1]       # '0'
            
            xxx = original_num_str[-3:] # '642'
            y = index_str               # '0'
            
            new_filename = f"IMG_{xxx}{y}_{suffix}{ext}" # 'IMG_6420_crop.png'
            
            old_path = os.path.join(target_dir, filename)
            new_path = os.path.join(target_dir, new_filename)
            
            if old_path != new_path:
                os.rename(old_path, new_path)

        except IndexError:
            continue # ファイル名が期待する形式でない場合はスキップ
        except Exception as e:
            continue # その他のエラーでも続行

def rename_processed_files(crop_dir: str, mask_dir: str, combined_dir: str):
    """
    処理済みのファイル名を指定されたルール（IMG_xxxY_suffix.png）に従って変更する。
    """
    _rename_files_in_folder(crop_dir, 'crop')
    _rename_files_in_folder(mask_dir, 'mask')
    _rename_files_in_folder(combined_dir, 'combined')


# -----------------------------------------------------------------
# [高レベル関数] パイプライン実行
# -----------------------------------------------------------------
def process_image_subdirectory(
    base_data_dir: str, 
    sub_folder: str, 
    model_path: str, 
    area_threshold: int = 50000
):
    """
    指定されたサブフォルダ（例: 'A'）に対して、
    セグメンテーションからリネームまでの一連の処理をすべて実行する。
    
    フォルダ構成は以下のようになっていることを前提とします:
    - {base_data_dir}/org/{sub_folder} (入力)
    - {base_data_dir}/crop/{sub_folder} (中間・出力)
    - {base_data_dir}/mask_org/{sub_folder} (中間)
    - {base_data_dir}/mask_to1/{sub_folder} (中間)
    - {base_data_dir}/mask_large/{sub_folder} (中間)
    - {base_data_dir}/mask_small/{sub_folder} (中間)
    - {base_data_dir}/mask/{sub_folder} (中間・出力)
    - {base_data_dir}/combined/{sub_folder} (出力)
    """
    
    # パス定義
    path_org = os.path.join(base_data_dir, 'org', sub_folder)
    path_crop = os.path.join(base_data_dir, 'crop', sub_folder)
    path_mask_org = os.path.join(base_data_dir, 'mask_org', sub_folder)
    path_mask_to1 = os.path.join(base_data_dir, 'mask_to1', sub_folder)
    path_mask_large = os.path.join(base_data_dir, 'mask_large', sub_folder)
    path_mask_small = os.path.join(base_data_dir, 'mask_small', sub_folder)
    path_mask_final = os.path.join(base_data_dir, 'mask', sub_folder)
    path_combined = os.path.join(base_data_dir, 'combined', sub_folder)

    # ステップ1: セグメンテーション
    segment_and_crop(
        input_image_dir=path_org,
        model_path=model_path,
        crop_dir=path_crop,
        mask_dir=path_mask_org
    )

    # ステップ2: 最大連結成分
    clean_masks_keep_largest(
        input_dir=path_mask_org,
        output_dir=path_mask_to1
    )

    # ステップ3: 面積フィルタリング
    filter_masks_by_area(
        source_mask_dir=path_mask_to1,
        large_mask_dir=path_mask_large,
        small_mask_dir=path_mask_small,
        crop_dir=path_crop,
        area_threshold=area_threshold
    )

    # ステップ4: 穴埋め
    fill_mask_holes(
        input_dir=path_mask_large,
        output_dir=path_mask_final
    )

    # ステップ5: マスク適用
    apply_mask_to_crop(
        crop_image_dir=path_crop,
        mask_image_dir=path_mask_final,
        output_dir=path_combined
    )

    # ステップ6: リネーム
    rename_processed_files(
        crop_dir=path_crop,
        mask_dir=path_mask_final,
        combined_dir=path_combined
    )

In [ ]:
import os
import time
from tqdm import tqdm # 実行側でプログレスバーを使いたい場合はこちらでimport

# -------------------------------------------------------------
# 1. 各種パスを指定
# -------------------------------------------------------------
# データセットの「親」フォルダ
BASE_DATA_DIR = '/home/data/1021_fasttest' 

# 学習済みモデルのパス
MODEL_PATH = '/home/YOLO/0708_maesyori/datasets/train/weights/best.pt'

# 処理対象のサブフォルダリスト
SUB_FOLDERS = ['A', 'B', 'C']

# 面積のしきい値
AREA_THRESHOLD = 50000

# -------------------------------------------------------------
# 2. パイプラインの実行
# -------------------------------------------------------------
print("画像処理パイプラインを開始します...")
start = time.time()
for folder_name in tqdm(SUB_FOLDERS, desc="Overall Progress"):
    print(f"\n--- Subfolder '{folder_name}' の処理を開始 ---")
    try:
        process_image_subdirectory(
            base_data_dir=BASE_DATA_DIR,
            sub_folder=folder_name,
            model_path=MODEL_PATH,
            area_threshold=AREA_THRESHOLD
        )
        print(f"--- Subfolder '{folder_name}' の処理が正常に完了しました。 ---")

    except FileNotFoundError as e:
        print(f"[警告] '{folder_name}' の処理中にエラー（フォルダ未検出）: {e}")
    except RuntimeError as e:
        print(f"[エラー] '{folder_name}' の処理中にランタイムエラー: {e}")
    except Exception as e:
        print(f"[致命的エラー] '{folder_name}' の処理中に予期せぬエラーが発生しました: {e}")
end = time.time()
elapsed = end - start
print(f"\n総処理時間: {elapsed:.2f} 秒")
print("\n===================================")
print("すべての処理が完了しました。")
print("===================================")

In [ ]:
import torch
from ultralytics import YOLO
from PIL import Image
import sys
import os
import cv2
import numpy as np
import glob
import shutil
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
# import functools # workers.pyに移動

# -----------------------------------------------------------------
# ワーカー関数を別ファイルからインポート
# -----------------------------------------------------------------
try:
    from workers import (
        _clean_mask_worker, 
        _filter_mask_worker, 
        _fill_mask_hole_worker, 
        _apply_mask_worker, 
        _rename_worker
    )
except ImportError:
    print("="*50, file=sys.stderr)
    print("エラー: 'workers.py' が見つかりません。", file=sys.stderr)
    print("このスクリプトと同じディレクトリに 'workers.py' を作成してください。", file=sys.stderr)
    print("="*50, file=sys.stderr)
    sys.exit(1)


# -----------------------------------------------------------------
# [ステップ1] セグメンテーションとクロップ (変更なし)
# -----------------------------------------------------------------
def segment_and_crop(input_image_dir: str, model_path: str, crop_dir: str, mask_dir: str):
    """
    YOLOモデルを使用して画像をセグメンテーションし、
    クロップ画像とマスク画像を保存する。
    """
    try:
        model = YOLO(model_path)
    except Exception as e:
        raise RuntimeError(f"モデルの読み込みに失敗しました: {model_path}. 詳細: {e}")

    os.makedirs(crop_dir, exist_ok=True)
    os.makedirs(mask_dir, exist_ok=True)

    allowed_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp']
    image_paths = []
    for ext in allowed_extensions:
        image_paths.extend(glob.glob(os.path.join(input_image_dir, ext)))

    if not image_paths:
        return 

    print(f"[ステップ1] {input_image_dir} のセグメンテーションを開始...")
    for image_path in tqdm(image_paths, desc="セグメンテーション"):
        try:
            img = Image.open(image_path).convert("RGB")
        except Exception as e:
            print(f"画像読み込み失敗: {image_path}, スキップします. {e}", file=sys.stderr)
            continue

        results = model(img, retina_masks=True)
        result = results[0]
        base_filename = os.path.splitext(os.path.basename(image_path))[0]

        if result.masks is not None and result.boxes is not None:
            num_objects = len(result.masks.data)
            original_img_np = np.array(img)

            for i in range(num_objects):
                try:
                    bbox_tensor = result.boxes.xyxy[i]
                    x1, y1, x2, y2 = map(int, bbox_tensor)
                    
                    cropped_object_rgb = original_img_np[y1:y2, x1:x2]
                    cropped_object_bgr = cv2.cvtColor(cropped_object_rgb, cv2.COLOR_RGB2BGR)
                    crop_filename = f"{base_filename}_object_{i}.png"
                    crop_path = os.path.join(crop_dir, crop_filename)
                    cv2.imwrite(crop_path, cropped_object_bgr)

                    full_mask_canvas = np.zeros(original_img_np.shape[:2], dtype=np.uint8)
                    mask_tensor = result.masks.data[i]
                    mask_np = mask_tensor.cpu().numpy().astype(np.uint8)
                    mask_resized = cv2.resize(mask_np, (original_img_np.shape[1], original_img_np.shape[0]))
                    full_mask_canvas[mask_resized > 0] = 255
                    
                    cropped_mask = full_mask_canvas[y1:y2, x1:x2]
                    
                    mask_filename = f"{base_filename}_mask_{i}.png"
                    mask_path = os.path.join(mask_dir, mask_filename)
                    cv2.imwrite(mask_path, cropped_mask)
                except Exception as e:
                    print(f"オブジェクト処理失敗: {image_path}, オブジェクト{i}. {e}", file=sys.stderr)
                    continue


# -----------------------------------------------------------------
# [ステップ2] マスクのクリーンアップ（ワーカーは workers.py に移動）
# -----------------------------------------------------------------
def parallel_clean_masks(input_dir: str, output_dir: str):
    """
    [並列実行] マスク画像の連結成分を解析し、最大の成分のみを残す。
    """
    os.makedirs(output_dir, exist_ok=True)
    
    try:
        image_files = [f for f in os.listdir(input_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    except FileNotFoundError:
        print(f"[ステップ2] 入力フォルダが見つかりません: {input_dir}", file=sys.stderr)
        return
    
    if not image_files:
        return

    tasks = [(os.path.join(input_dir, f), os.path.join(output_dir, f)) for f in image_files]

    print(f"[ステップ2] {input_dir} のクリーンアップを開始...")
    with Pool(processes=cpu_count()) as pool:
        list(tqdm(pool.imap_unordered(_clean_mask_worker, tasks), total=len(tasks), desc="最大連結成分"))

# -----------------------------------------------------------------
# [ステップ3] 面積によるフィルタリング（ワーカーは workers.py に移動）
# -----------------------------------------------------------------
def parallel_filter_masks_by_area(
    source_mask_dir: str, 
    large_mask_dir: str, 
    small_mask_dir: str, 
    crop_dir: str, 
    area_threshold: int
):
    """
    [並列実行] マスク画像の面積に基づき画像を振り分け、
    小さいマスクに対応するクロップ画像を削除する。
    """
    os.makedirs(large_mask_dir, exist_ok=True)
    os.makedirs(small_mask_dir, exist_ok=True)

    try:
        filenames = [
            f for f in os.listdir(source_mask_dir) 
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp'))
        ]
    except FileNotFoundError:
        print(f"[ステップ3] 元フォルダが見つかりません: {source_mask_dir}", file=sys.stderr)
        return

    if not filenames:
        return

    tasks = [
        (
            os.path.join(source_mask_dir, f), 
            large_mask_dir, 
            small_mask_dir, 
            crop_dir, 
            area_threshold, 
            f
        ) for f in filenames
    ]

    print(f"[ステップ3] {source_mask_dir} の面積フィルタリングを開始...")
    with Pool(processes=cpu_count()) as pool:
        list(tqdm(pool.imap_unordered(_filter_mask_worker, tasks), total=len(tasks), desc="面積フィルタ"))

# -----------------------------------------------------------------
# [ステップ4] マスクの穴埋め（ワーカーは workers.py に移動）
# -----------------------------------------------------------------
def parallel_fill_mask_holes(input_dir: str, output_dir: str):
    """
    [並列実行] マスク画像内の「真の穴」（画像端に接していない穴）を塗りつぶす。
    """
    os.makedirs(output_dir, exist_ok=True)

    try:
        image_files = [f for f in os.listdir(input_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    except FileNotFoundError:
        print(f"[ステップ4] 入力フォルダが見つかりません: {input_dir}", file=sys.stderr)
        return
    
    if not image_files:
        return

    tasks = [(os.path.join(input_dir, f), os.path.join(output_dir, f)) for f in image_files]

    print(f"[ステップ4] {input_dir} の穴埋めを開始...")
    with Pool(processes=cpu_count()) as pool:
        list(tqdm(pool.imap_unordered(_fill_mask_hole_worker, tasks), total=len(tasks), desc="穴埋め"))

# -----------------------------------------------------------------
# [ステップ5] マスクの適用（ワーカーは workers.py に移動）
# -----------------------------------------------------------------
def parallel_apply_mask_to_crop(crop_image_dir: str, mask_image_dir: str, output_dir: str):
    """
    [並列実行] クロップ画像にマスク画像を適用し、背景を黒にする。
    """
    os.makedirs(output_dir, exist_ok=True)

    try:
        crop_files = [
            f for f in os.listdir(crop_image_dir) 
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]
    except FileNotFoundError:
        print(f"[ステップ5] クロップ画像フォルダが見つかりません: {crop_image_dir}", file=sys.stderr)
        return

    if not crop_files:
        return
    
    tasks = [
        (
            os.path.join(crop_image_dir, f), 
            mask_image_dir, 
            output_dir, 
            f
        ) for f in crop_files
    ]
    
    print(f"[ステップ5] {crop_image_dir} へのマスク適用を開始...")
    with Pool(processes=cpu_count()) as pool:
        list(tqdm(pool.imap_unordered(_apply_mask_worker, tasks), total=len(tasks), desc="マスク適用"))

# -----------------------------------------------------------------
# [ステップ6] ファイル名の変更（ワーカーは workers.py に移動）
# -----------------------------------------------------------------
def _parallel_rename_folder(target_dir: str, suffix: str, pool: Pool):
    """ [ヘルパー] 1つのフォルダのリネームを並列実行する """
    try:
        filenames = [
            f for f in os.listdir(target_dir) 
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]
    except FileNotFoundError:
        print(f"[ステップ6] フォルダが見つかりません: {target_dir}", file=sys.stderr)
        return

    if not filenames:
        return
        
    tasks = [(target_dir, f, suffix) for f in filenames]
    
    list(tqdm(pool.imap_unordered(_rename_worker, tasks), total=len(tasks), desc=f"リネーム ({suffix})"))

def parallel_rename_processed_files(crop_dir: str, mask_dir: str, combined_dir: str):
    """
    [並列実行] 処理済みのファイル名を指定されたルールに従って変更する。
    """
    print(f"[ステップ6] {crop_dir} 等のリネームを開始...")
    with Pool(processes=cpu_count()) as pool:
        _parallel_rename_folder(crop_dir, 'crop', pool)
        _parallel_rename_folder(mask_dir, 'mask', pool)
        _parallel_rename_folder(combined_dir, 'combined', pool)


# -----------------------------------------------------------------
# [高レベル関数] パイプライン実行 (変更なし)
# -----------------------------------------------------------------
def process_image_subdirectory(
    base_data_dir: str, 
    sub_folder: str, 
    model_path: str, 
    area_threshold: int = 50000
):
    """
    指定されたサブフォルダ（例: 'A'）に対して、
    セグメンテーションからリネームまでの一連の処理をすべて実行する。
    (CPU処理は並列化)
    """
    
    print(f"\n--- 処理開始: {sub_folder} ---")
    
    # パス定義
    path_org = os.path.join(base_data_dir, 'org', sub_folder)
    path_crop = os.path.join(base_data_dir, 'crop', sub_folder)
    path_mask_org = os.path.join(base_data_dir, 'mask_org', sub_folder)
    path_mask_to1 = os.path.join(base_data_dir, 'mask_to1', sub_folder)
    path_mask_large = os.path.join(base_data_dir, 'mask_large', sub_folder)
    path_mask_small = os.path.join(base_data_dir, 'mask_small', sub_folder)
    path_mask_final = os.path.join(base_data_dir, 'mask', sub_folder)
    path_combined = os.path.join(base_data_dir, 'combined', sub_folder)

    # ステップ1: セグメンテーション (GPU - 変更なし)
    segment_and_crop(
        input_image_dir=path_org,
        model_path=model_path,
        crop_dir=path_crop,
        mask_dir=path_mask_org
    )

    # ステップ2: 最大連結成分 (CPU - 並列化)
    parallel_clean_masks(
        input_dir=path_mask_org,
        output_dir=path_mask_to1
    )

    # ステップ3: 面積フィルタリング (CPU - 並列化)
    parallel_filter_masks_by_area(
        source_mask_dir=path_mask_to1,
        large_mask_dir=path_mask_large,
        small_mask_dir=path_mask_small,
        crop_dir=path_crop,
        area_threshold=area_threshold
    )

    # ステップ4: 穴埋め (CPU - 並列化)
    parallel_fill_mask_holes(
        input_dir=path_mask_large,
        output_dir=path_mask_final
    )

    # ステップ5: マスク適用 (CPU - 並列化)
    parallel_apply_mask_to_crop(
        crop_image_dir=path_crop,
        mask_image_dir=path_mask_final,
        output_dir=path_combined
    )

    # ステップ6: リネーム (CPU - 並列化)
    parallel_rename_processed_files(
        crop_dir=path_crop,
        mask_dir=path_mask_final,
        combined_dir=path_combined
    )
    
    print(f"--- 処理完了: {sub_folder} ---")


# -----------------------------------------------------------------
# 実行 (マルチプロセスには if __name__ == "__main__": が必須)
# -----------------------------------------------------------------
if __name__ == "__main__":
    
    # --- パラメータ設定 (例) ---
    BASE_DIR = '/home/data/1021_fasttest' 
    MODEL_PATH = '/home/YOLO/0708_maesyori/datasets/train/weights/best.pt'
    AREA_THRESHOLD = 50000
    
    # 処理対象のサブフォルダ (A, B, Cなど)
    TARGET_FOLDERS = ['A', 'B', 'C'] 
    
    # --- 実行ループ ---
    start = time.time()
    try:
        for folder in TARGET_FOLDERS:
            process_image_subdirectory(
                base_data_dir=BASE_DIR,
                sub_folder=folder,
                model_path=MODEL_PATH,
                area_threshold=AREA_THRESHOLD
            )
    except Exception as e:
        print(f"\nパイプライン全体で致命的なエラーが発生しました: {e}", file=sys.stderr)
        sys.exit(1)
    
    end = time.time()
    elapsed = end - start
    print(f"\n総処理時間: {elapsed:.2f} 秒")

In [ ]:
#フォルダごとにマルチプロセスで並列実行するコード例
import multiprocessing
import time
import sys
import os

# -----------------------------------------------------------------
# 1. pipeline_full.py から「ワーカー関数」をインポート
# -----------------------------------------------------------------
try:
    # 警告: 
    # ノートブック環境では、pipeline_full.py を変更した場合、
    # このセルを実行する前に「カーネルの再起動」が必要です。
    from pipeline_full import process_image_subdirectory
except ImportError:
    print("="*50, file=sys.stderr)
    print("エラー: 'pipeline_full.py' が見つかりません。", file=sys.stderr)
    print("このノートブックと同じディレクトリに 'pipeline_full.py' を作成してください。", file=sys.stderr)
    print("="*50, file=sys.stderr)
    # エラーが発生した場合、ダミーの関数を定義して NameError を防ぐ
    def process_image_subdirectory(*args, **kwargs):
        print("ダミー関数が実行されました。インポートに失敗しています。", file=sys.stderr)


# -----------------------------------------------------------------
# 2. 実行
# (マルチプロセスは if __name__ == "__main__": が必須)
# -----------------------------------------------------------------
if __name__ == "__main__":
    
    # ★★★ マルチプロセスの設定 ★★★
    try:
        multiprocessing.set_start_method('spawn', force=True)
        print("マルチプロセスの開始メソッドを 'spawn' に設定しました。")
    except RuntimeError:
        print("マルチプロセスの開始メソッドは既に設定されています。")

    # --- パラメータ設定 (例) ---
    BASE_DIR = '/home/data/1021_fasttest' 
    MODEL_PATH = '/home/YOLO/0708_maesyori/datasets/train/weights/best.pt'
    AREA_THRESHOLD = 50000
    
    # 処理対象のサブフォルダ (A, B, Cなど)
    TARGET_FOLDERS = ['A', 'B', 'C'] 
    
    # --- 実行ループ (ここを並列化) ---
    
    print("フォルダ単位の並列処理を開始します...")
    print(f"対象フォルダ: {TARGET_FOLDERS}")
    
    overall_start_time = time.time()
    
    processes = [] # プロセスを管理するリスト
    
    for folder in TARGET_FOLDERS:
        # フォルダごとにプロセスを作成
        # target にはインポートした関数を指定
        p = multiprocessing.Process(
            target=process_image_subdirectory, 
            args=(
                BASE_DIR, 
                folder, 
                MODEL_PATH, 
                AREA_THRESHOLD
            )
        )
        processes.append(p)
        p.start() # プロセスを開始

    # すべてのプロセスの終了を待つ
    for p in processes:
        p.join()

    overall_end_time = time.time()
    print(f"\nすべての処理が完了しました。 (総所要時間: {overall_end_time - overall_start_time:.2f}秒)")

In [2]:
import multiprocessing
import time
import sys
import os
from functools import partial # ★ 1. partial をインポート

# -----------------------------------------------------------------
# 1. pipeline_full.py から「ワーカー関数」をインポート
# -----------------------------------------------------------------
try:
    from pipeline_full import process_image_subdirectory
except ImportError:
    print("="*50, file=sys.stderr)
    print("エラー: 'pipeline_full.py' が見つかりません。", file=sys.stderr)
    print("="*50, file=sys.stderr)
    def process_image_subdirectory(*args, **kwargs):
        print("ダミー関数が実行されました。インポートに失敗しています。", file=sys.stderr)


# -----------------------------------------------------------------
# 2. 実行
# -----------------------------------------------------------------
if __name__ == "__main__":
    
    # ★★★ マルチプロセスの設定 ★★★
    try:
        multiprocessing.set_start_method('spawn', force=True)
        print("マルチプロセスの開始メソッドを 'spawn' に設定しました。")
    except RuntimeError:
        print("マルチプロセスの開始メソッドは既に設定されています。")

    # --- パラメータ設定 (例) ---
    BASE_DIR = '/home/data/1021_fasttest' 
    MODEL_PATH = '/home/YOLO/0708_maesyori/datasets/train/weights/best.pt'
    AREA_THRESHOLD = 50000
    TARGET_FOLDERS = ['A', 'B', 'C'] 
    
    # ★ 2. VRAMから判断した「並列数の上限」を定義
    NUM_WORKERS = 2
    
    # --- 実行ループ (Pool を使って並列化) ---
    
    print(f"フォルダ単位の並列処理を {NUM_WORKERS} 並列で開始します...")
    print(f"対象フォルダ: {TARGET_FOLDERS}")
    
    overall_start_time = time.time()
    
    # ★ 3. 複数の引数を渡すため、partial で関数をラップ
    # (process_image_subdirectory は引数が多いため)
    process_func = partial(
        process_image_subdirectory,
        BASE_DIR,
        model_path=MODEL_PATH,
        area_threshold=AREA_THRESHOLD
        # 最後の引数 'sub_folder' は pool.map から渡される
    )
    
    # ★ 4. Pool を使って、上限(NUM_WORKERS)付きで実行
    with multiprocessing.Pool(processes=NUM_WORKERS) as pool:
        # pool.map は TARGET_FOLDERS の各要素 ('A', 'B', 'C') を
        # process_func の最後の引数 (sub_folder) として順番に渡して実行します
        # 2並列の場合、まず 'A' と 'B' が実行され、
        # どちらかが終わったら 'C' が実行されます。
        pool.map(process_func, TARGET_FOLDERS)

    # ★ 5. 元の for ループ (p.start(), p.join()) は不要なので削除

    overall_end_time = time.time()
    print(f"\nすべての処理が完了しました。 (総所要時間: {overall_end_time - overall_start_time:.2f}秒)")

マルチプロセスの開始メソッドを 'spawn' に設定しました。
フォルダ単位の並列処理を 2 並列で開始します...
対象フォルダ: ['A', 'B', 'C']
FlashAttention is not available on this device. Using scaled_dot_product_attention instead.
FlashAttention is not available on this device. Using scaled_dot_product_attention instead.
--- [13051] 処理開始: A ---
[13051] ステップ1 開始: /home/data/1021_fasttest/org/A
--- [13050] 処理開始: B ---
[13050] ステップ1 開始: /home/data/1021_fasttest/org/B


0: 640x480 42 shiitakes, 34.1ms
Speed: 3.2ms preprocess, 34.1ms inference, 96.0ms postprocess per image at shape (1, 3, 640, 480)
0: 640x480 52 shiitakes, 35.3ms
Speed: 3.4ms preprocess, 35.3ms inference, 98.7ms postprocess per image at shape (1, 3, 640, 480)

0: 640x480 42 shiitakes, 3.1ms
Speed: 1.3ms preprocess, 3.1ms inference, 14.7ms postprocess per image at shape (1, 3, 640, 480)

0: 640x480 53 shiitakes, 3.1ms
Speed: 1.5ms preprocess, 3.1ms inference, 18.7ms postprocess per image at shape (1, 3, 640, 480)

0: 640x480 42 shiitakes, 3.1ms
Speed: 1.4ms preprocess, 3.1ms inf

GPUマルチプロセスのための確認

In [ ]:
# check_vram.py
import time
import torch
import pipeline_full # 作成したモジュールをインポート

# ---- パラメータ設定 ----
BASE_DIR = '/home/data/1021_fasttest' 
MODEL_PATH = '/home/YOLO/0708_maesyori/datasets/train/weights/best.pt'
TARGET_FOLDER = 'A' # ★★★ まずは1つだけで試す ★★★
AREA_THRESHOLD = 50000

print(f"--- VRAM使用量チェック開始: フォルダ {TARGET_FOLDER} ---")
print(f"使用するモデル: {MODEL_PATH}")
print(f"GPUが利用可能か: {torch.cuda.is_available()}")

start_time = time.time()

try:
    # 1プロセス（1フォルダ）だけ処理を実行
    pipeline_full.process_image_subdirectory(
        base_data_dir=BASE_DIR, 
        sub_folder=TARGET_FOLDER, 
        model_path=MODEL_PATH, 
        area_threshold=AREA_THRESHOLD
    )

except Exception as e:
    print(f"処理中にエラーが発生しました: {e}")

finally:
    end_time = time.time()
    print(f"--- チェック完了 (所要時間: {end_time - start_time:.2f}秒) ---")

    if torch.cuda.is_available():
        # PyTorchが認識しているピーク時のVRAM使用量（参考値）
        peak_vram = torch.cuda.max_memory_allocated() / (1024**2) # MB単位
        print(f"PyTorchによるピークVRAM使用量（推定）: {peak_vram:.2f} MB")
        torch.cuda.empty_cache() # メモリ解放